# Pairs Trading — Mean Reversion on Correlated Stocks
Classic stat-arb: find cointegrated pairs, trade the spread when it deviates from the mean.
Testing on tech pairs: MSFT/GOOGL, AAPL/MSFT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import coint

plt.style.use('dark_background')

In [ ]:
# Download data
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']
data = yf.download(tickers, start='2020-01-01', end='2024-12-31', progress=False)['Close']
data = data.dropna()
print(f'Period: {data.index[0].date()} → {data.index[-1].date()}, {len(data)} bars')

In [ ]:
# Cointegration test for all pairs
from itertools import combinations

results = []
for t1, t2 in combinations(tickers, 2):
    score, pvalue, _ = coint(data[t1], data[t2])
    results.append({'pair': f'{t1}/{t2}', 'coint_stat': round(score, 3), 'p_value': round(pvalue, 4)})

pairs_df = pd.DataFrame(results).sort_values('p_value')
print('Cointegration Test Results:')
print(pairs_df.to_string(index=False))
print(f'\nBest pair: {pairs_df.iloc[0]["pair"]} (p={pairs_df.iloc[0]["p_value"]})')

In [ ]:
# Trade the best pair
best = pairs_df.iloc[0]['pair']
t1, t2 = best.split('/')

# Compute hedge ratio via OLS
from numpy.polynomial.polynomial import polyfit
b, a = np.polyfit(data[t2], data[t1], 1)
spread = data[t1] - b * data[t2]

# Z-score of spread
lookback = 30
spread_mean = spread.rolling(lookback).mean()
spread_std = spread.rolling(lookback).std()
z_score = (spread - spread_mean) / spread_std

print(f'Hedge ratio: {b:.4f}')
print(f'Spread mean: {spread.mean():.2f}, std: {spread.std():.2f}')

In [ ]:
# Plot spread and z-score
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(spread.index, spread, color='#22d3ee', linewidth=0.8)
ax1.axhline(spread.mean(), color='#71717a', linestyle='--')
ax1.set_title(f'Spread: {t1} - {b:.2f} × {t2}', fontsize=13)
ax1.grid(alpha=0.2)

ax2.plot(z_score.index, z_score, color='#a78bfa', linewidth=0.8)
ax2.axhline(1.5, color='#ef4444', linestyle='--', alpha=0.5)
ax2.axhline(-1.5, color='#34d399', linestyle='--', alpha=0.5)
ax2.axhline(0, color='#71717a', linestyle='--', alpha=0.3)
ax2.fill_between(z_score.index, -1.5, 1.5, alpha=0.05, color='white')
ax2.set_title('Z-Score of Spread', fontsize=13)
ax2.set_ylabel('Z-Score')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Backtest pairs strategy
ENTRY_Z = 1.5
EXIT_Z = 0.0
capital = 100_000
cash = capital
position = 0  # 1 = long spread, -1 = short spread
equity = []
trades = []

for i in range(lookback, len(z_score)):
    z = z_score.iloc[i]
    spread_val = spread.iloc[i]
    
    if z < -ENTRY_Z and position == 0:
        position = 1
        entry_spread = spread_val
    elif z > ENTRY_Z and position == 0:
        position = -1
        entry_spread = spread_val
    elif abs(z) < EXIT_Z and position != 0:
        pnl = (spread_val - entry_spread) * position * 100
        cash += pnl
        trades.append({'pnl': pnl, 'z_entry': z})
        position = 0
    
    equity.append(cash)

print(f'Total trades: {len(trades)}')
print(f'Win rate: {sum(1 for t in trades if t["pnl"] > 0) / len(trades) * 100:.1f}%')
print(f'Total PnL: ${sum(t["pnl"] for t in trades):,.0f}')
print(f'Final equity: ${equity[-1]:,.0f}')